In [ ]:
#U-NET
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/pred_subseasonal_model_u_tp_20260831.nc"
clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/PR/U-NET/2026/TWO_quintile_clim-20260831-tp.nc"
output_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测"

output_file_q1 = os.path.join(output_dir, "S-fc-UNet-20260831-tp-q1.nc")
output_file_q2 = os.path.join(output_dir, "S-fc-UNet-20260831-tp-q2.nc")

# ===============================
# 确保输出目录存在
# ===============================
os.makedirs(output_dir, exist_ok=True)

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算单日五分位概率函数
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []
    for i in range(5):
        if i == 0:
            p = (forecast < clim_quintiles[0])
        elif i == 4:
            p = (forecast >= clim_quintiles[3])
        else:
            p = (forecast >= clim_quintiles[i-1]) & (forecast < clim_quintiles[i])
        # 删除旧坐标，避免 concat 冲突
        p = p.drop_vars([v for v in p.coords if v not in ["latitude","longitude"]])
        prob_list.append(p.astype(np.float32))
    
    prob = xr.concat(prob_list, dim="quintile", coords="minimal", compat="override")
    prob = prob.assign_coords(quintile=quintiles)
    prob = prob / prob.sum(dim="quintile")  # 确保每个格点五分位和为1
    return prob

# ===============================
# 第一周 Q1
# ===============================
pr_clim_q1 = clim_ds["q1"].isel(time=0)  # 气候分位数 q1
week_days_q1 = ["tp_mon", "tp_wed", "tp_fri", "tp_sun"]

daily_probs_q1 = []
for day in week_days_q1:
    daily_forecast = forecast_ds[day].isel(time=0)
    prob = compute_quintile_prob(daily_forecast, pr_clim_q1)
    daily_probs_q1.append(prob)

final_q1 = xr.concat(daily_probs_q1, dim="day").mean(dim="day")
final_q1.name = "tp_quintile_prob1"

# 安全保存 Q1
temp_file = output_file_q1 + ".tmp"
final_q1.to_netcdf(temp_file)
os.rename(temp_file, output_file_q1)
print("✅ 第一周四日平均 Q1 已保存到：", output_file_q1)

# ===============================
# 第二周 Q2
# ===============================
pr_clim_q2 = clim_ds["q2"].isel(time=0)  # 气候分位数 q2
week_days_q2 = ["tp_tue_next", "tp_thu_next", "tp_sat_next"]

daily_probs_q2 = []
for day in week_days_q2:
    daily_forecast = forecast_ds[day].isel(time=0)
    prob = compute_quintile_prob(daily_forecast, pr_clim_q2)
    daily_probs_q2.append(prob)

final_q2 = xr.concat(daily_probs_q2, dim="day").mean(dim="day")
final_q2.name = "tp_quintile_prob2"

# 安全保存 Q2
temp_file = output_file_q2 + ".tmp"
final_q2.to_netcdf(temp_file)
os.rename(temp_file, output_file_q2)
print("✅ 第二周三日平均 Q2 已保存到：", output_file_q2)


In [ ]:
#U-NET
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/S-fc-UNet-20260831-tp-q1.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/S-fc-UNet-20260831-tp-q2.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

# 坐标和quintile
lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values  # array([0.2, 0.4, 0.6, 0.8, 1.0])

# 读取变量
data_q1 = ds_q1['tp_quintile_prob1']
data_q2 = ds_q2['tp_quintile_prob2']

# 创建图形，2行5列
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# 统一色标范围
vmin, vmax = 0, 1
cmap = 'YlGnBu'  # 适合降水概率的渐变色

for i in range(5):
    # q1 行
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i), cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2 行
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i), cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 添加 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
#fig.colorbar(im, cax=cbar_ax, label='Probability')

#fig.suptitle("TP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

#fig.savefig("tp_quintile_probabilities_20260831.png",
#            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# UPU-NET
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/33subseasonal_model_u-tp_best_acc_20260831.nc"
clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/PR/UPU-NET/2026/TWO_quintile_clim-20260831-tp.nc"
output_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测"

output_file_q1 = os.path.join(output_dir, "UPU-NET-20260831-tp-q1.nc")
output_file_q2 = os.path.join(output_dir, "UPU-NET-20260831-tp-q2.nc")

# ===============================
# 确保输出目录存在
# ===============================
os.makedirs(output_dir, exist_ok=True)

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算单日五分位概率函数
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []
    for i in range(5):
        if i == 0:
            p = (forecast < clim_quintiles[0])
        elif i == 4:
            p = (forecast >= clim_quintiles[3])
        else:
            p = (forecast >= clim_quintiles[i-1]) & (forecast < clim_quintiles[i])
        # 删除旧坐标，避免 concat 冲突
        p = p.drop_vars([v for v in p.coords if v not in ["latitude","longitude"]])
        prob_list.append(p.astype(np.float32))
    
    prob = xr.concat(prob_list, dim="quintile", coords="minimal", compat="override")
    prob = prob.assign_coords(quintile=quintiles)
    prob = prob / prob.sum(dim="quintile")  # 确保每个格点五分位和为1
    return prob

# ===============================
# 第一周 Q1
# ===============================
pr_clim_q1 = clim_ds["q1"].isel(time=0)  # 气候分位数 q1
week_days_q1 = ["tp_mon", "tp_wed", "tp_fri", "tp_sun"]

daily_probs_q1 = []
for day in week_days_q1:
    daily_forecast = forecast_ds[day].isel(time=0)
    prob = compute_quintile_prob(daily_forecast, pr_clim_q1)
    daily_probs_q1.append(prob)

final_q1 = xr.concat(daily_probs_q1, dim="day").mean(dim="day")
final_q1.name = "tp_quintile_prob1"

# 安全保存 Q1
temp_file = output_file_q1 + ".tmp"
final_q1.to_netcdf(temp_file)
os.rename(temp_file, output_file_q1)
print("✅ 第一周四日平均 Q1 已保存到：", output_file_q1)

# ===============================
# 第二周 Q2
# ===============================
pr_clim_q2 = clim_ds["q2"].isel(time=0)  # 气候分位数 q2
week_days_q2 = ["tp_tue_next", "tp_thu_next", "tp_sat_next"]

daily_probs_q2 = []
for day in week_days_q2:
    daily_forecast = forecast_ds[day].isel(time=0)
    prob = compute_quintile_prob(daily_forecast, pr_clim_q2)
    daily_probs_q2.append(prob)

final_q2 = xr.concat(daily_probs_q2, dim="day").mean(dim="day")
final_q2.name = "tp_quintile_prob2"

# 安全保存 Q2
temp_file = output_file_q2 + ".tmp"
final_q2.to_netcdf(temp_file)
os.rename(temp_file, output_file_q2)
print("✅ 第二周三日平均 Q2 已保存到：", output_file_q2)

In [ ]:
# UPU-NET
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/UPU-NET-20260831-tp-q1.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/UPU-NET-20260831-tp-q2.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

# 坐标和quintile
lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values  # array([0.2, 0.4, 0.6, 0.8, 1.0])

# 读取变量
data_q1 = ds_q1['tp_quintile_prob1']
data_q2 = ds_q2['tp_quintile_prob2']

# 创建图形，2行5列
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# 统一色标范围
vmin, vmax = 0, 1
cmap = 'YlGnBu'  # 适合降水概率的渐变色

for i in range(5):
    # q1 行
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i), cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2 行
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i), cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 添加 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

#fig.suptitle("TP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

#fig.savefig("tp_quintile_probabilities_20260831.png",
#            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# SA
import os
import numpy as np
import xarray as xr

# ===============================
# 路径设置
# ===============================
forecast_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/33SpatialAttentionResNet_TP_20260831.nc"
clim_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/气候四分位数/PR/SA/2026/TWO_quintile_clim-20260831-tp.nc"
output_dir = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测"

output_file_q1 = os.path.join(output_dir, "SA-20260831-tp-q1.nc")
output_file_q2 = os.path.join(output_dir, "SA-20260831-tp-q2.nc")

# ===============================
# 确保输出目录存在
# ===============================
os.makedirs(output_dir, exist_ok=True)

# ===============================
# 读取数据
# ===============================
forecast_ds = xr.open_dataset(forecast_file)
clim_ds = xr.open_dataset(clim_file)

# 五分位坐标
quintiles = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

# ===============================
# 计算单日五分位概率函数
# ===============================
def compute_quintile_prob(forecast, clim_quintiles):
    prob_list = []
    for i in range(5):
        if i == 0:
            p = (forecast < clim_quintiles[0])
        elif i == 4:
            p = (forecast >= clim_quintiles[3])
        else:
            p = (forecast >= clim_quintiles[i-1]) & (forecast < clim_quintiles[i])
        # 删除旧坐标，避免 concat 冲突
        p = p.drop_vars([v for v in p.coords if v not in ["latitude","longitude"]])
        prob_list.append(p.astype(np.float32))
    
    prob = xr.concat(prob_list, dim="quintile", coords="minimal", compat="override")
    prob = prob.assign_coords(quintile=quintiles)
    prob = prob / prob.sum(dim="quintile")  # 确保每个格点五分位和为1
    return prob

# ===============================
# 第一周 Q1
# ===============================
pr_clim_q1 = clim_ds["q1"].isel(time=0)  # 气候分位数 q1
week_days_q1 = ["tp_mon", "tp_wed", "tp_fri", "tp_sun"]

daily_probs_q1 = []
for day in week_days_q1:
    daily_forecast = forecast_ds[day].isel(time=0)
    prob = compute_quintile_prob(daily_forecast, pr_clim_q1)
    daily_probs_q1.append(prob)

final_q1 = xr.concat(daily_probs_q1, dim="day").mean(dim="day")
final_q1.name = "tp_quintile_prob1"

# 安全保存 Q1
temp_file = output_file_q1 + ".tmp"
final_q1.to_netcdf(temp_file)
os.rename(temp_file, output_file_q1)
print("✅ 第一周四日平均 Q1 已保存到：", output_file_q1)

# ===============================
# 第二周 Q2
# ===============================
pr_clim_q2 = clim_ds["q2"].isel(time=0)  # 气候分位数 q2
week_days_q2 = ["tp_tue_next", "tp_thu_next", "tp_sat_next"]

daily_probs_q2 = []
for day in week_days_q2:
    daily_forecast = forecast_ds[day].isel(time=0)
    prob = compute_quintile_prob(daily_forecast, pr_clim_q2)
    daily_probs_q2.append(prob)

final_q2 = xr.concat(daily_probs_q2, dim="day").mean(dim="day")
final_q2.name = "tp_quintile_prob2"

# 安全保存 Q2
temp_file = output_file_q2 + ".tmp"
final_q2.to_netcdf(temp_file)
os.rename(temp_file, output_file_q2)
print("✅ 第二周三日平均 Q2 已保存到：", output_file_q2)

In [ ]:
# SA
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/SA-20260831-tp-q1.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/SA-20260831-tp-q2.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

# 坐标和quintile
lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values  # array([0.2, 0.4, 0.6, 0.8, 1.0])

# 读取变量
data_q1 = ds_q1['tp_quintile_prob1']
data_q2 = ds_q2['tp_quintile_prob2']

# 创建图形，2行5列
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# 统一色标范围
vmin, vmax = 0, 1
cmap = 'YlGnBu'  # 适合降水概率的渐变色

for i in range(5):
    # q1 行
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i), cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2 行
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i), cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 添加 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

#fig.suptitle("TP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

#fig.savefig("tp_quintile_probabilities_20260831.png",
#            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 第一个时刻
import os
import xarray as xr
import numpy as np

# === 文件目录与模型文件 ===
folder = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/"

model_files = [
    "S-fc-UNet-20260831-tp-q1.nc",
    "UPU-NET-20260831-tp-q1.nc",
    "SA-20260831-tp-q1.nc",
]

# === 读取所有模型的数据并累加 ===
sum_probs = None

for file in model_files:
    path = os.path.join(folder, file)
    ds = xr.open_dataset(path)
    
    # 提取五分位概率变量: (quintile, lat, lon)
    prob = ds['tp_quintile_prob1']  # shape: (5, 121, 240)
    
    if sum_probs is None:
        sum_probs = prob
    else:
        sum_probs += prob

# === 计算均值 ===
mean_probs = sum_probs / len(model_files)

# === 归一化：保证每个格点上5个分位数和为1 ===
mean_probs_normalized = mean_probs / mean_probs.sum(dim='quintile')

# === 保存为新文件 ===
output_path = os.path.join(folder, "20260831-1-ensemble_quintile_prob_tp_mean.nc")
mean_probs_normalized.to_dataset(name="ensemble_tp_quintile_prob1").to_netcdf(output_path)

print(f"已保存至: {output_path}")

In [ ]:

# 第二个时刻
import os
import xarray as xr
import numpy as np

# === 文件目录与模型文件 ===
folder = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/"

model_files = [
    "S-fc-UNet-20260831-tp-q2.nc",
    "UPU-NET-20260831-tp-q2.nc",
    "SA-20260831-tp-q2.nc",
]

# === 读取所有模型的数据并累加 ===
sum_probs = None

for file in model_files:
    path = os.path.join(folder, file)
    ds = xr.open_dataset(path)
    
    # 提取五分位概率变量: (quintile, lat, lon)
    prob = ds['tp_quintile_prob2']  # shape: (5, 121, 240)
    
    if sum_probs is None:
        sum_probs = prob
    else:
        sum_probs += prob

# === 计算均值 ===
mean_probs = sum_probs / len(model_files)

# === 归一化：保证每个格点上5个分位数和为1 ===
mean_probs_normalized = mean_probs / mean_probs.sum(dim='quintile')

# === 保存为新文件 ===
output_path = os.path.join(folder, "20260831-2-ensemble_quintile_prob_tp_mean.nc")
mean_probs_normalized.to_dataset(name="ensemble_tp_quintile_prob2").to_netcdf(output_path)

print(f"已保存至: {output_path}")

In [ ]:
# 全球绘图
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/20260831-1-ensemble_quintile_prob_tp_mean.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/20260831-2-ensemble_quintile_prob_tp_mean.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

# 坐标和quintile
lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values  # array([0.2, 0.4, 0.6, 0.8, 1.0])

# 读取变量
data_q1 = ds_q1['ensemble_tp_quintile_prob1']
data_q2 = ds_q2['ensemble_tp_quintile_prob2']

# 创建图形，2行5列
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# 统一色标范围
vmin, vmax = 0, 1
cmap = 'YlGnBu'  # 适合降水概率的渐变色

for i in range(5):
    # q1 行
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i), cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2 行
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i), cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 添加 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

fig.suptitle("TP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

fig.savefig("tp_quintile_probabilities_20260831.png",
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 中国绘图
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/20260831-1-ensemble_quintile_prob_tp_mean.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/20260831-2-ensemble_quintile_prob_tp_mean.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values

data_q1 = ds_q1['ensemble_tp_quintile_prob1']
data_q2 = ds_q2['ensemble_tp_quintile_prob2']

# 创建图形
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

vmin, vmax = 0, 1
cmap = 'YlGnBu'

for i in range(5):
    # q1 行
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([73, 135, 5, 54], crs=ccrs.PlateCarree())
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2 行
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([73, 135, 5, 54], crs=ccrs.PlateCarree())
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 统一 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

fig.suptitle("TP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

fig.savefig("CHINAtp_quintile_probabilities_20260831.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 西非绘图
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# 文件路径
file_q1 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/20260831-1-ensemble_quintile_prob_tp_mean.nc"
file_q2 = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 预测/20260831-2-ensemble_quintile_prob_tp_mean.nc"

# 读取数据
ds_q1 = xr.open_dataset(file_q1)
ds_q2 = xr.open_dataset(file_q2)

lat = ds_q1.latitude.values
lon = ds_q1.longitude.values
quintiles = ds_q1.quintile.values

data_q1 = ds_q1['ensemble_tp_quintile_prob1']
data_q2 = ds_q2['ensemble_tp_quintile_prob2']

# 创建图形
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(35, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.3, hspace=0.3)

vmin, vmax = 0, 1
cmap = 'YlGnBu'

for i in range(5):
    # q1 行
    ax = axes[0, i]
    im = ax.pcolormesh(lon, lat, data_q1.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([-25, 40, -10, 72], crs=ccrs.PlateCarree())
    ax.set_title(f"Prob1 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    # q2 行
    ax = axes[1, i]
    im = ax.pcolormesh(lon, lat, data_q2.isel(quintile=i),
                       cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    ax.add_feature(cfeature.COASTLINE, edgecolor='black')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([-25, 40, -10, 72], crs=ccrs.PlateCarree())
    ax.set_title(f"Prob2 - Quintile {quintiles[i]:.1f}")
    ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

# 统一 colorbar
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Probability')

fig.suptitle("TP Ensemble Quintile Probabilities 20260831&20250922", fontsize=20)

fig.savefig("WESTtp_quintile_probabilities_20260831.png", dpi=300, bbox_inches='tight')
plt.show()